# Fase 2 · Notebook 01 — Preprocesamiento y partición

Proyecto de aula de Inteligencia Artificial · Grupo 04
**Predicción de reingresos hospitalarios a 30 días en pacientes con diabetes**

---

Este notebook deja los datos listos para modelar. Hace cinco cosas, en este orden:

1. **Carga** el archivo original tal como viene de UCI.
2. **Construye la cohorte**: retira los egresos a los que la pregunta no se puede aplicar.
3. **Define la variable objetivo** `y`.
4. **Crea variables nuevas** (ingeniería de características) y retira las que no sirven.
5. **Parte los datos en 60/20/20 separando por paciente**, no por fila.

Todo lo que produce queda guardado en `data/processed/`, y los notebooks siguientes parten de ahí.

> **Nada de lo que se hace aquí mira el conjunto de prueba.** Las decisiones de limpieza
> dependen solo de la estructura del archivo, no del desenlace.

## 1. Librerías y rutas

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

# Las rutas se construyen a partir de la carpeta del proyecto, para que el
# notebook funcione en cualquier computador sin editar nada.
BASE      = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW       = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Carpeta del proyecto:", BASE)

Carpeta del proyecto: C:\Users\ASUS\Documents\UPC\proyecto-reingresos


## 2. Carga de los datos

El archivo usa `?` para indicar ausencia, así que se le dice a pandas que lo interprete
como dato faltante. `low_memory=False` evita que pandas adivine mal el tipo de las columnas
mixtas de diagnósticos.

In [2]:
raw = pd.read_csv(RAW / "diabetic_data.csv", na_values=["?"], low_memory=False)

print(f"Filas:    {raw.shape[0]:>7,}")
print(f"Columnas: {raw.shape[1]:>7,}")
print(f"Pacientes distintos: {raw.patient_nbr.nunique():>6,}")

assert raw.shape == (101766, 50), "El archivo original debe tener 101766 filas y 50 columnas"
print("\nVerificacion OK: el archivo coincide con el descrito en la Fase 1.")

Filas:    101,766
Columnas:      50
Pacientes distintos: 71,518

Verificacion OK: el archivo coincide con el descrito en la Fase 1.


## 3. Construcción de la cohorte

**Cohorte** = los registros que sí entran al estudio, después de apartar aquellos a los que
la pregunta no se les puede aplicar.

La pregunta del proyecto es *"¿este paciente volverá al hospital antes de 30 días?"*. Hay dos
situaciones en las que esa pregunta no tiene sentido:

| Situación | Por qué se excluye |
|---|---|
| El paciente **falleció** | No puede reingresar. Dejarlo marcado como "no reingresó" le enseñaría al modelo que era un paciente de bajo riesgo, que es justo lo contrario de la realidad. |
| El paciente pasó a **cuidados paliativos** (*hospice*) | Está en un proceso de fin de vida, no en un seguimiento normal tras el alta. |

Esos casos se identifican por el código de egreso `discharge_disposition_id`. El archivo
`IDs_mapping.csv` traduce los códigos; los que corresponden a fallecimiento o cuidados
paliativos son **11, 13, 14, 19, 20 y 21**.

Es el mismo criterio del estudio original del dataset (Strack et al., 2014).

In [3]:
EXCLUIR = [11, 13, 14, 19, 20, 21]

# Traducimos los codigos para dejar constancia de que se esta excluyendo
import csv, io as _io
texto = (RAW / "IDs_mapping.csv").read_text(encoding="utf-8", errors="ignore")
bloque = texto.split("discharge_disposition_id,description")[1].split("admission_source_id")[0]
descripcion = {int(f[0]): f[1].strip()
               for f in csv.reader(_io.StringIO(bloque))
               if len(f) == 2 and f[0].strip().isdigit()}

detalle = pd.DataFrame({
    "codigo": EXCLUIR,
    "significado": [descripcion.get(c, "?") for c in EXCLUIR],
    "registros": [(raw.discharge_disposition_id == c).sum() for c in EXCLUIR],
})
display(detalle)
print(f"Total de registros excluidos: {detalle.registros.sum():,}")

,codigo,significado,registros
0,11,Expired,1642
1,13,Hospice / home,399
2,14,Hospice / medical facility,372
3,19,"Expired at home. Medicaid only, hospice.",8
4,20,"Expired in a medical facility. Medicaid only, ...",2
5,21,"Expired, place unknown. Medicaid only, hospice.",0


Total de registros excluidos: 2,423


In [4]:
cohorte = raw[~raw.discharge_disposition_id.isin(EXCLUIR)].reset_index(drop=True)

print(f"Antes  : {len(raw):>7,} hospitalizaciones | {raw.patient_nbr.nunique():>6,} pacientes")
print(f"Despues: {len(cohorte):>7,} hospitalizaciones | {cohorte.patient_nbr.nunique():>6,} pacientes")

assert len(cohorte) == 99343, "La cohorte debe tener 99343 hospitalizaciones"
assert cohorte.patient_nbr.nunique() == 69990, "La cohorte debe tener 69990 pacientes"
print("\nVerificacion OK: coincide con las cifras reportadas en la Fase 1.")

Antes  : 101,766 hospitalizaciones | 71,518 pacientes
Despues:  99,343 hospitalizaciones | 69,990 pacientes

Verificacion OK: coincide con las cifras reportadas en la Fase 1.


## 4. Variable objetivo

La columna original `readmitted` tiene tres valores: `<30`, `>30` y `NO`. El proyecto se
ocupa del **reingreso temprano**, así que se recodifica en una variable binaria:

```
y = 1   cuando readmitted es "<30"
y = 0   cuando readmitted es ">30" o "NO"
```

In [5]:
cohorte["y"] = (cohorte.readmitted == "<30").astype(int)

resumen = (cohorte.readmitted.value_counts()
           .rename_axis("readmitted").to_frame("registros"))
resumen["porcentaje"] = (100 * resumen.registros / len(cohorte)).round(2)
display(resumen)

positivos = int(cohorte.y.sum())
prevalencia = cohorte.y.mean()
print(f"Positivos (y=1): {positivos:>6,}  ->  {100*prevalencia:.2f} %")
print(f"Negativos (y=0): {len(cohorte)-positivos:>6,}  ->  {100*(1-prevalencia):.2f} %")
print(f"\nRazon de desbalance: {(1-prevalencia)/prevalencia:.1f} negativos por cada positivo")

,registros,porcentaje
readmitted,,
NO,52527,52.87
>30,35502,35.74
<30,11314,11.39


Positivos (y=1): 11,314  ->  11.39 %
Negativos (y=0): 88,029  ->  88.61 %

Razon de desbalance: 7.8 negativos por cada positivo


> ⚠️ **Consecuencia del desbalance.** Un modelo que siempre responda "no reingresa" acertaría
> el **88,61 %** de las veces y **no detectaría ni un solo reingreso**. Por eso la *exactitud*
> (accuracy) no sirve como criterio de selección en este proyecto, y la métrica principal será
> **Average Precision**, que sí penaliza no detectar los positivos.

## 5. Ingeniería de características

**Ingeniería de características** significa *crear variables nuevas a partir de las que ya
existen*, para que el modelo tenga información más útil.

Ejemplo: el archivo trae `num_medications` (cuántos medicamentos) y `time_in_hospital`
(cuántos días). Por separado dicen poco. Pero su cociente —medicamentos por día— es una señal
de **intensidad del tratamiento**: 20 medicamentos en 2 días no es lo mismo que 20 en 10 días.

Estas son las 12 variables que se crean y por qué:

| Variable nueva | Cómo se calcula | Qué aporta |
|---|---|---|
| `edad_num` | punto medio del intervalo de `age` | convierte `[60-70)` en 65, un número con el que se puede calcular |
| `servicios_previos` | consultas + urgencias + ingresos del año anterior | uso total de servicios previo |
| `tuvo_ingreso_previo` | 1 si hubo ingresos previos | la Fase 1 mostró que el reingreso sube de 8,5 % a 26,8 % según este dato |
| `meds_por_dia` | `num_medications / time_in_hospital` | intensidad del tratamiento |
| `labs_por_dia` | `num_lab_procedures / time_in_hospital` | intensidad del estudio diagnóstico |
| `num_meds_activos` | cuántos de los 23 medicamentos no están en "No" | complejidad del esquema farmacológico |
| `hubo_cambio_med` | 1 si `change` es "Ch" | hubo ajuste del tratamiento |
| `a1c_medida` | 1 si se midió la hemoglobina glicosilada | Strack et al. (2014) encontraron que medirla se asocia al reingreso |
| `glu_medida` | 1 si se midió glucosa en suero | igual que la anterior |
| `grupo_diag_1/2/3` | agrupa los códigos ICD-9 en 9 categorías clínicas | `diag_1` tiene más de 700 códigos distintos: inservibles tal cual, útiles agrupados |

### Sobre la agrupación de diagnósticos

Los códigos ICD-9 son números como `250.83` o `428`. Hay cientos y cada uno aparece pocas
veces, así que usarlos directamente generaría cientos de columnas casi vacías. Se agrupan en
las nueve categorías clínicas del estudio original:

| Categoría | Rango ICD-9 |
|---|---|
| Circulatorio | 390–459, 785 |
| Respiratorio | 460–519, 786 |
| Digestivo | 520–579, 787 |
| Diabetes | 250.xx |
| Lesiones | 800–999 |
| Musculoesquelético | 710–739 |
| Genitourinario | 580–629, 788 |
| Neoplasias | 140–239 |
| Otros | el resto, incluidos los códigos V y E |

In [6]:
# Los 23 medicamentos que trae el archivo
MEDICAMENTOS = [
    'metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
    'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin',
    'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone',
]

EDAD_PUNTO_MEDIO = {'[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,'[40-50)':45,
                    '[50-60)':55,'[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95}

def agrupar_diagnostico(codigo):
    """Traduce un codigo ICD-9 a una de las nueve categorias clinicas."""
    if pd.isna(codigo):
        return "Desconocido"
    texto = str(codigo)
    if texto.startswith(("V", "E")):      # codigos suplementarios: no son numericos
        return "Otros"
    try:
        v = float(texto)
    except ValueError:
        return "Otros"
    entero = int(v)
    if 390 <= v <= 459 or entero == 785: return "Circulatorio"
    if 460 <= v <= 519 or entero == 786: return "Respiratorio"
    if 520 <= v <= 579 or entero == 787: return "Digestivo"
    if entero == 250:                     return "Diabetes"
    if 800 <= v <= 999:                   return "Lesiones"
    if 710 <= v <= 739:                   return "Musculoesqueletico"
    if 580 <= v <= 629 or entero == 788: return "Genitourinario"
    if 140 <= v <= 239:                   return "Neoplasias"
    return "Otros"

print("Funciones listas.")

Funciones listas.


In [7]:
df = cohorte.copy()

df["edad_num"]            = df.age.map(EDAD_PUNTO_MEDIO)
df["servicios_previos"]   = df.number_outpatient + df.number_emergency + df.number_inpatient
df["tuvo_ingreso_previo"] = (df.number_inpatient > 0).astype(int)
df["meds_por_dia"]        = df.num_medications / df.time_in_hospital
df["labs_por_dia"]        = df.num_lab_procedures / df.time_in_hospital
df["num_meds_activos"]    = (df[MEDICAMENTOS] != "No").sum(axis=1)
df["hubo_cambio_med"]     = (df.change == "Ch").astype(int)
df["a1c_medida"]          = (df.A1Cresult.notna()      & (df.A1Cresult != "None")).astype(int)
df["glu_medida"]          = (df.max_glu_serum.notna()  & (df.max_glu_serum != "None")).astype(int)

for i in (1, 2, 3):
    df[f"grupo_diag_{i}"] = df[f"diag_{i}"].map(agrupar_diagnostico)

NUEVAS = ["edad_num","servicios_previos","tuvo_ingreso_previo","meds_por_dia","labs_por_dia",
          "num_meds_activos","hubo_cambio_med","a1c_medida","glu_medida",
          "grupo_diag_1","grupo_diag_2","grupo_diag_3"]

print(f"Se crearon {len(NUEVAS)} variables nuevas.\n")
display(df[["edad_num","servicios_previos","meds_por_dia","labs_por_dia","num_meds_activos"]]
        .describe().round(2))

Se crearon 12 variables nuevas.



,edad_num,servicios_previos,meds_por_dia,labs_por_dia,num_meds_activos
count,99343.00,99343.00,99343.00,99343.00,99343.00
mean,65.74,1.20,5.07,14.32,1.19
std,15.95,2.29,3.81,11.94,0.92
min,5.00,0.00,0.08,0.07,0.00
25%,55.00,0.00,2.60,6.50,1.00
50%,65.00,0.00,4.00,11.00,1.00
75%,75.00,2.00,6.17,18.33,2.00
max,95.00,80.00,42.00,92.00,6.00


In [8]:
# Como quedo la agrupacion del diagnostico principal
tabla = df.grupo_diag_1.value_counts().rename_axis("categoria").to_frame("registros")
tabla["porcentaje"] = (100 * tabla.registros / len(df)).round(2)
display(tabla)
print(f"diag_1 tenia {df.diag_1.nunique()} codigos distintos -> ahora son {df.grupo_diag_1.nunique()} categorias.")

,registros,porcentaje
categoria,,
Circulatorio,29681,29.88
Otros,17793,17.91
Respiratorio,13934,14.03
Digestivo,9333,9.39
Diabetes,8661,8.72
Lesiones,6853,6.90
Genitourinario,5002,5.04
Musculoesqueletico,4935,4.97
Neoplasias,3131,3.15


diag_1 tenia 715 codigos distintos -> ahora son 10 categorias.


## 6. Columnas que se retiran

No todas las columnas sirven. Estas se retiran, cada una por una razón distinta:

| Columna | Razón |
|---|---|
| `weight` | **96,86 % de ausentes**. Imputarla para casi toda la base exigiría supuestos imposibles de sostener. |
| `payer_code` | 39,56 % de ausentes y depende del sistema de aseguramiento estadounidense, no del estado del paciente. |
| `examide`, `citoglipton` | **Constantes**: valen `"No"` en las 101.766 filas. Una columna que no cambia no puede distinguir a nadie. |
| `race` | Se reserva para revisar diferencias entre subgrupos; **se excluye del modelo principal** según lo definido en la Fase 1. |
| `diag_1`, `diag_2`, `diag_3` | Reemplazadas por `grupo_diag_1/2/3`. |
| `age` | Reemplazada por `edad_num`. |
| `readmitted` | Reemplazada por `y`. Dejarla sería darle la respuesta al modelo. |
| `encounter_id` | Identificador, no una característica. |

`patient_nbr` **no se usa como predictor**, pero se conserva aparte porque es lo que permite
partir los datos por paciente.

> `medical_specialty` tiene 49 % de ausentes pero **sí se conserva**: la ausencia misma puede
> ser informativa (depende de cómo registró cada hospital) y se tratará como una categoría más.

In [9]:
RETIRAR = ["weight", "payer_code", "examide", "citoglipton", "race",
           "diag_1", "diag_2", "diag_3", "age", "readmitted", "encounter_id"]

X = df.drop(columns=RETIRAR + ["y", "patient_nbr"])
y = df["y"]
grupos = df["patient_nbr"]

# Estas tres columnas son numeros, pero NO son cantidades: son codigos
# administrativos. Hay que marcarlas como categoricas (ver explicacion abajo).
CODIGOS = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
X[CODIGOS] = X[CODIGOS].astype(str)

numericas   = [c for c in X.columns if c not in CODIGOS and pd.api.types.is_numeric_dtype(X[c])]
categoricas = [c for c in X.columns if c not in numericas]

print(f"Columnas retiradas: {len(RETIRAR)}")
print(f"Caracteristicas finales: {X.shape[1]}")
print(f"   numericas   : {len(numericas):>2}")
print(f"   categoricas : {len(categoricas):>2}  (incluye los 3 codigos administrativos)")
print(f"\nFilas: {len(X):,}  |  Objetivo y: {y.sum():,} positivos")

Columnas retiradas: 11
Caracteristicas finales: 50
   numericas   : 17
   categoricas : 33  (incluye los 3 codigos administrativos)

Filas: 99,343  |  Objetivo y: 11,314 positivos


### ⚠️ Por qué los tres códigos administrativos se tratan como categorías

`admission_type_id`, `discharge_disposition_id` y `admission_source_id` **están escritos como
números, pero no son cantidades**. Son etiquetas:

| Código | `discharge_disposition_id` significa |
|---|---|
| 1 | Dado de alta a casa |
| 3 | Trasladado a un centro de cuidados prolongados |
| 6 | Alta a casa con servicio de salud domiciliario |
| 25 | No facturado |

Si se dejaran como números, el modelo asumiría que **25 es "más" que 6**, y que la distancia
entre 1 y 3 es la misma que entre 3 y 5. **Eso es falso**: son categorías sin orden ni distancia,
igual que si los códigos fueran colores.

Es el mismo error que se comete al numerar las clases en vez de usar *one-hot*: se inventa un
orden que no existe en los datos.

Por eso se convierten a texto con `.astype(str)`, para que el codificador las trate como lo que
son. Tienen 8, 26 y 17 valores distintos: una cantidad perfectamente manejable.

## 7. Partición 60 / 20 / 20 **por paciente**

Esta es la decisión más importante del notebook.

**El problema:** en la cohorte hay 99.343 hospitalizaciones pero solo 69.990 personas.
16.341 pacientes tienen más de una hospitalización, y uno llega a tener **40**.

Si se repartieran las filas al azar, la misma persona quedaría en entrenamiento **y** en
prueba. El modelo la habría visto mientras aprendía, y al evaluarlo acertaría —pero no por
saber predecir, sino por **reconocerla**. Los resultados saldrían falsamente buenos.

**La solución:** `StratifiedGroupKFold` reparte **grupos completos**. Un paciente entero va a
un solo conjunto, con todas sus hospitalizaciones. Además, *stratified* significa que mantiene
la misma proporción de positivos en los tres conjuntos.

Se parte en 5 pliegues del 20 % cada uno y se reparten así:

```
pliegues 0, 1, 2  ->  entrenamiento (60 %)
pliegue  4        ->  validacion    (20 %)
pliegue  3        ->  prueba        (20 %)
```

> **Sin semilla aleatoria.** `StratifiedGroupKFold` sin `shuffle` es determinista: produce
> siempre el mismo resultado en cualquier computador. Eso hace el proyecto completamente
> reproducible y, además, da exactamente los mismos tamaños reportados en la Fase 1.

In [10]:
particionador = StratifiedGroupKFold(n_splits=5)
pliegues = [indices for _, indices in particionador.split(X, y, groups=grupos)]

idx_train = np.concatenate(pliegues[:3])
idx_val   = pliegues[4]
idx_test  = pliegues[3]

df["conjunto"] = ""
df.loc[idx_train, "conjunto"] = "entrenamiento"
df.loc[idx_val,   "conjunto"] = "validacion"
df.loc[idx_test,  "conjunto"] = "prueba"

resumen = []
for nombre, idx in [("entrenamiento", idx_train), ("validacion", idx_val), ("prueba", idx_test)]:
    sub = df.iloc[idx]
    resumen.append({
        "conjunto": nombre,
        "hospitalizaciones": len(sub),
        "porcentaje": round(100 * len(sub) / len(df), 1),
        "pacientes": sub.patient_nbr.nunique(),
        "positivos": int(sub.y.sum()),
        "prevalencia_%": round(100 * sub.y.mean(), 2),
    })
display(pd.DataFrame(resumen))

,conjunto,hospitalizaciones,porcentaje,pacientes,positivos,prevalencia_%
0,entrenamiento,59605,60.0,41988,6789,11.39
1,validacion,19868,20.0,13982,2262,11.39
2,prueba,19870,20.0,14020,2263,11.39


### Verificaciones

Tres comprobaciones automáticas. Si alguna falla, el notebook se detiene: es preferible que
reviente aquí a que un error silencioso contamine todos los resultados.

In [11]:
pac_train = set(df.iloc[idx_train].patient_nbr)
pac_val   = set(df.iloc[idx_val].patient_nbr)
pac_test  = set(df.iloc[idx_test].patient_nbr)

print("1. Ningun paciente aparece en dos conjuntos")
print(f"   entrenamiento ∩ validacion : {len(pac_train & pac_val)}")
print(f"   entrenamiento ∩ prueba     : {len(pac_train & pac_test)}")
print(f"   validacion    ∩ prueba     : {len(pac_val   & pac_test)}")
assert not (pac_train & pac_val) and not (pac_train & pac_test) and not (pac_val & pac_test)

print("\n2. No se perdio ni se duplico ninguna fila")
total = len(idx_train) + len(idx_val) + len(idx_test)
print(f"   {len(idx_train):,} + {len(idx_val):,} + {len(idx_test):,} = {total:,}  (cohorte: {len(df):,})")
assert total == len(df) and len(set(np.concatenate([idx_train, idx_val, idx_test]))) == len(df)

print("\n3. La proporcion de positivos se mantiene en los tres conjuntos")
for nombre, idx in [("entrenamiento", idx_train), ("validacion", idx_val), ("prueba", idx_test)]:
    print(f"   {nombre:<14} {100*df.iloc[idx].y.mean():.2f} %")
assert abs(df.iloc[idx_train].y.mean() - df.iloc[idx_test].y.mean()) < 0.01

print("\nLas tres verificaciones pasaron.")

1. Ningun paciente aparece en dos conjuntos
   entrenamiento ∩ validacion : 0
   entrenamiento ∩ prueba     : 0
   validacion    ∩ prueba     : 0

2. No se perdio ni se duplico ninguna fila
   59,605 + 19,868 + 19,870 = 99,343  (cohorte: 99,343)

3. La proporcion de positivos se mantiene en los tres conjuntos
   entrenamiento  11.39 %
   validacion     11.39 %
   prueba         11.39 %

Las tres verificaciones pasaron.


## 8. Guardar los resultados

Se guardan dos archivos en `data/processed/`:

- **`cohorte_procesada.csv`** — la cohorte con las variables nuevas, la columna `y`,
  `patient_nbr` y la etiqueta de conjunto.
- **`columnas.json`** — qué columnas son numéricas y cuáles categóricas, para que el
  notebook siguiente arme los pipelines sin tener que repetir la clasificación.

> Estos archivos **no se suben al repositorio** (están en `.gitignore`): se regeneran
> ejecutando este notebook, que es la forma correcta de versionar datos derivados.

In [12]:
import json

salida = df[["patient_nbr", "conjunto", "y"] + list(X.columns)]
salida.to_csv(PROCESSED / "cohorte_procesada.csv", index=False)

(PROCESSED / "columnas.json").write_text(json.dumps({
    "numericas": numericas,
    "categoricas": categoricas,
    "objetivo": "y",
    "grupo": "patient_nbr",
}, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Guardado: {PROCESSED / 'cohorte_procesada.csv'}")
print(f"   {salida.shape[0]:,} filas x {salida.shape[1]} columnas")
print(f"Guardado: {PROCESSED / 'columnas.json'}")

Guardado: C:\Users\ASUS\Documents\UPC\proyecto-reingresos\data\processed\cohorte_procesada.csv
   99,343 filas x 53 columnas
Guardado: C:\Users\ASUS\Documents\UPC\proyecto-reingresos\data\processed\columnas.json


---

## Resumen de lo que hizo este notebook

| | |
|---|---|
| Archivo original | 101.766 hospitalizaciones · 71.518 pacientes |
| Excluidos (fallecimiento / cuidados paliativos) | 2.423 |
| **Cohorte** | **99.343 hospitalizaciones · 69.990 pacientes** |
| Positivos (`y = 1`) | 11.314 — **11,39 %** |
| Variables nuevas creadas | 12 |
| Columnas retiradas | 11 |
| Características finales | 50 |
| Partición | 59.605 / 19.868 / 19.870 — **por paciente** |
| Pacientes compartidos entre conjuntos | **0** |

**Siguiente:** `02_modelos_baseline.ipynb` — clasificador mayoritario y regresión logística.